<a href="https://colab.research.google.com/github/SajaFawagreh/SYSC4415/blob/main/W2026/Assignments/A3/SYSC4415_W26_A3_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Assignment 3  
## 80 Mandatory + 20 Bonus marks

## General Instructions:

This Assignment can be done **in a group of two or individually**.

**YOU MUST JOIN A GROUP ON BRIGHTSPACE TO SUBMIT EVEN IF YOU WORK ON YOUR OWN.**

Please state it explicitly at the beginning of the assignment.

You need only one submission if it's group work.

Please print out values when asked using Python's print() function with f-strings where possible.

Submit your **saved notebook with all the outputs** to Brightspace, but ensure it will produce correct outputs upon restarting and click "runtime" → "run all" with clean outputs. Ensure your notebook displays all answers correctly.

```It is highly recommended that you setup your environment locally and use your laptops or desktop computers to work on this assignment rather than Google Colab as you need proper control of your environment.```

## Your Submission MUST contain your signature at the bottom.

### **Objective:** In this assignment, we build a ReAct agent that facilitates **ML operations and model evaluation**.

This assignment is heavily based on [Tutorial 10](https://github.com/emslaboratory/SYSC4415/blob/master/W2026/Tutorials/T10/T10_SimpleAgent.ipynb).

**Submission:** Submit your Notebook with saved outputs as a *.ipynb* file that adopts this naming convention: ***SYSC4415_W26_A3_NameLastname.ipynb*** on *Brightspace*. No other submission (e.g., through email) will be accepted. (Example file name: ```SYSC4415_W26_A3_NameLastname.ipynb``` or ```SYSC4415_W26_A3_Student1_Student2.ipynb```) The notebook MUST contain saved outputs.

**Runtime tips:**
Agentic programming and API calling can be easily done locally and moved to Colab in the final stages, depending on the implementation of your tools and ML tasks you want to run.

In [1]:
# Name: Saja Fawagreh
# Student Number: 101217326

# Name:Joseph Marques
# Student Number: 101218139

In [2]:
# Libraries to install - leave this code block blank if this does not apply to you
# Please add a brief comment on why you need the library and what it does


# Imports

Some basic libraries you need are imported here. Make sure you include whatever library you need in this entire notebook in the code block below.

If you are using any library that requires installation, please paste the installation command here.
Leave the code block below if you are not installing any libraries.

In [3]:
# Libraries you might need
# General
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# For pre-processing
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

# For modeling
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torchsummary

# For metrics
from sklearn.metrics import  accuracy_score
from sklearn.metrics import  precision_score
from sklearn.metrics import  recall_score
from sklearn.metrics import  f1_score
from sklearn.metrics import  classification_report
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import  roc_auc_score
from sklearn.metrics import confusion_matrix


# Use the SAME Python interpreter that's running this notebook
# to install the package — avoids version mismatch between
# system Python and the notebook kernel
import subprocess, sys
subprocess.check_call([
    sys.executable,       # e.g. /usr/bin/python3.11
    "-m", "pip",
    "install",
    "--upgrade",          # ensure we get the latest version
    "--quiet",            # suppress verbose pip output
    "llm-connector"
])

# llm_connector.cli contains the one-time project scaffolding tool
from llm_connector.cli import init_project

# Creates this folder structure in your working directory:
#   llm-connector/
#     conf/        ← provider config files go here
#     logs/        ← session logs written here automatically
#     .env.template ← copy this to .env and add your API keys
#
# Safe to call multiple times — skips creation if folder already exists
init_project()


# Agent
from llm_connector import chat_completion, cleanup_resources
from dataclasses import dataclass
import re
import json
import time
from typing import Dict, List, Optional


[!] Directory already exists: /content/llm-connector
    Use --force to overwrite, or delete it manually.


# Task 1: Registration and API Activation (5 marks)

For this particular assignment, we will be using OpenRouter or GroqCloud for LLM inference. This task helps you initiate your first LLM call.

Create a free account on [groq.com](https://groq.com/) or/and [openrouter.com](https://openrouter.ai/) and generate an API Key. Both providers maintain free LLM access.

If you ran the previous cell, then at this point you have the following file structure:

```
├── llm-connector
│   ├── .env.template
│   ├── conf
│   │   ├── llm.yaml
│   │   ├── logs.yaml
│   │   ├── override.yaml.template
│   │   └── security.yaml
│   └── logs
│       └── pricing
└── SYSC4415_W26_A3.ipynb
```
You need to rename your .env.template file to ".env" In the template you will see, and you need the first two unless you want to use other providers:

```
OPENROUTER_API_KEY=YOUR-API-KEY
GROQ_API_KEY=YOUR-API-KEY
GOOGLE_API_KEY=YOUR-API-KEY
OPENAI_API_KEY=YOUR-API-KEY
ANTHROPIC_API_KEY=YOUR-API-KEY
```
Add your API key value to the corresponding variables (OPENROUTER_API_KEY or GROQ_API_KEY).

#### ```You must restart notebook Kernel to start using the .env file```

This Assignement uses open-source ```llm-connector``` ```(pip install llm-connector)``` python module that provides a unified interface to prompt LLMs from different providers (OpenRouter, Groq, OpenAI, Anthropic, Google). It was developed by [Igor Bogdanov](mailto:igorbogdanov@cmail.carleton.ca) and is used in his research projects.

#### If you find the ```llm-connector``` module useful, feel free to star it on GitHub: https://github.com/isbogdanov/llm_connector).

**Please report the issues and feel free to use the module in your agentic projects.**


In [17]:
# Q1a (3 marks)
# Select Provider and Model, set hyper-parameters and test message to test that completion works.
# Hint: Use Tutorial 10 and Providers' Documentation
# Explain each parameter and how each value change influences the LLM's output.
from groq import Groq
import os
import time
from llm_connector import chat_completion, cleanup_resources

# Load API key (from .env)
from dotenv import load_dotenv
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

# In general you can use any model and provider you like if you have access.
# Make sure you understand that some models hide their reasoning.
# Recommended: llama-3.3-70b-versatile - for Groq, and nvidia/nemotron-3-nano-30b-a3b:free - for OpenRouter
# Try not to send to many requests in order to stay within you free tier limit.

PROVIDER = "groq"  # Specifies the API provider used to access the model (Groq offers fast LLM inference)
MODEL = "llama-3.3-70b-versatile"  # The selected large language model; a powerful model for high-quality responses
TEMPERATURE = 0.2  # Controls randomness: lower values (0–0.3) make the output more deterministic and focused
TOP_P = 0.7  # Controls diversity via nucleus sampling: lower values limit the output to more probable tokens
MAX_TOKENS = 1024  # Maximum number of tokens in the response; higher values allow longer outputs

messages = [
    {"role": "user", "content": "Hello! What is happening? What model are you?"}
]

response, p, c, t, latency = chat_completion(
    messages,
    provider=(PROVIDER, MODEL),
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS
)

print(response, p, c, t, latency)


Hello! I'm an AI designed to assist and communicate with users in a helpful and informative way. I'm a type of language model, specifically a large language model (LLM) developed by Meta AI. My primary function is to understand and respond to natural language inputs, providing accurate and relevant information on a wide range of topics.

I'm a text-based AI model, which means I can understand and respond to text-based inputs, but I don't have the ability to see, hear, or interact with the physical world. My responses are generated based on patterns and associations in the data I was trained on, and I'm constantly learning and improving my responses based on user interactions.

I don't have personal experiences, emotions, or consciousness like humans do, but I'm designed to provide helpful and informative responses to your questions and engage in conversation to the best of my abilities. So, what's on your mind? What would you like to talk about? 46 193 239 1.1015737520001494


In [18]:
# Q1b (2 marks)
# Prompt the model using the user role about anything different from the tutorial.

# YOUR ANSWER GOES HERE
messages = [
    {"role": "user", "content": "Give me 3 advantages of using AI in healthcare."}
]

response, p, c, t, latency = chat_completion(
    messages,
    provider=(PROVIDER, MODEL),
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS
)

print(response, p, c, t, latency)

1. **Improved Diagnostic Accuracy**: AI can analyze large amounts of medical data, including images, lab results, and patient histories, to help doctors make more accurate diagnoses. AI-powered algorithms can detect patterns and anomalies that may be missed by human clinicians, leading to better patient outcomes.

2. **Enhanced Patient Care and Personalization**: AI can help personalize treatment plans for patients by analyzing their unique characteristics, medical histories, and genetic profiles. This can lead to more effective treatment and better patient engagement, as patients receive care that is tailored to their specific needs.

3. **Streamlined Clinical Workflows and Efficiency**: AI can automate routine administrative tasks, such as data entry, billing, and insurance claims, freeing up clinicians to focus on more complex and high-value tasks. AI can also help optimize resource allocation, reduce wait times, and improve patient flow, leading to increased efficiency and producti

# Task 2: Agent Implementation (5 marks)

This task contains an implementation of the agent from Tutorial 10. The idea of this task is to make sure you understand how basic LLM-Agent works.


In conversational AI, prompting involves three key roles:
1. **the system role** which provides the foundational instructions and constraints as well as sets the agent's behavior and capabilities;
2. **the user role** delivers the actual queries or commands;
3. **the assistant role** generates contextual, step-by-step responses following the system's guidelines.

This structured approach ensures consistent, controlled interactions where the agent maintains its defined behavior while responding to user needs, with each role serving a specific purpose in the conversation flow. If you dont set system role the model will use its defaults.

In [25]:
# Q2a: (5 marks)
# Explain how agent implementation works, providing comments line by line. And answer the question.
# You are not allowed to copy comments from Tutorial 10.
# This paper might be helpful: https://react-lm.github.io/

# Explain how agent implementation works
"""
This agent tracks the conversation using a messages list. It starts with a system prompt that defines the agent’s behavior.
Each time the user sends a message, it is appended to the conversation history. The full message history is then passed to the language model through chat_completion().
The model generates a response based on the entire conversation, not just the latest message. The assistant’s response is then added to the history,
so the next interaction includes both previous user inputs and assistant replies. This creates a multi-turn conversational agent with memory.
"""
# Why do we append the whole conversation for each subsequent call
"""
We include the full conversation so the model has context. Language models are stateless by default,
meaning they do not remember previous messages unless those messages are included again in the next API call.
By sending the entire conversation each time, the model can understand what was previously asked,
maintain consistency, and generate replies that fit the ongoing discussion.
"""
# and what are the potential negative consequences of this?
"""
1. It increases token usage, which can make the API more expensive.
2. It increases latency because longer inputs take more time to process.
3. If the conversation becomes too long, it may exceed the model’s context window, causing older messages to be dropped or truncated.
4. Including too much history can sometimes confuse the model or cause it to focus on irrelevant earlier details.
"""

@dataclass
class AgentState:
    messages: List[Dict[str, str]] # Stores the conversation history as a list of role-content message dictionaries
    system_prompt: str # Stores the original system instruction that defines the agent's behavior

class ML_Agent:

    # Initializes the agent with a system prompt that sets rules, behavior, and context
    def __init__(self, system_prompt: str):
        self.state = AgentState(
            messages=[{"role": "system", "content": system_prompt}], # Starts conversation history with the system message
            system_prompt=system_prompt, # Saves the system prompt separately for reference
        )

    # Adds a new message to the conversation history
    def add_message(self, role: str, content: str) -> None:
        self.state.messages.append({"role": role, "content": content})

    # Sends the full conversation history to the language model and gets a reply
    def execute(self) -> str:
        response, p, c, t, latency = chat_completion(
            self.state.messages, # Passes all previous messages so the model keeps conversational context
            provider=(PROVIDER, MODEL), # Selects which provider and model to use
            temperature=TEMPERATURE, # Controls randomness of the generated response
            top_p=TOP_P, # Controls how diverse the token selection can be
            max_tokens=MAX_TOKENS # Limits the maximum response length
        )

        # Handles the case where the model does not return a valid response
        if response is None:
            response = "[No response from model]"

        return response # Returns the generated assistant reply

    # Allows the object to be used like a function when given a user message
    def __call__(self, message: str) -> str:
        self.add_message("user", message) # Adds the new user input to memory
        result = self.execute() # Generates a reply using the current conversation state
        self.add_message("assistant", result) # Stores the assistant reply so future calls remember it
        return result # Returns the final assistant response

# Task 3: Tools (29 marks)

Tools are specialized functions that enable AI agents to perform specific actions beyond their inherent capabilities, such as retrieving information, performing calculations, or manipulating data. Agents use tools to decompose complex reasoning into observable steps, extend their knowledge beyond training data, maintain state across interactions, and provide transparency in their decision-making process, ultimately allowing them to solve problems they couldn't tackle through reasoning alone.

Essentially, tools are just callback functions invoked by the agent at the appropriate time during the execution loop.

You need to plan your tools for each particular task your agent is expected to solve.
The Model Evaluation Agent we are building should be able to evaluate the model from the model pool on the specific dataset.

Datasets to use: Penguins, Iris, CIFAR-10

You should be able to tell the agent what to do and watch it display the output of the tools' execution, similar to that in Tutorial 10.

User Prompt examples you should be able to give to your agent and expect it to fulfill the task:
- **Evaluate Linear Regression Model on Iris Dataset**
- **Train a logistic regression model on the Iris dataset**
- **Load the Penguins dataset and preprocess it.**
- **Train a decision tree model on the Penguins dataset and evaluate it.**
- **Load the CIFAR-10 dataset and train Mini-ResNet CNN, visualize results**

Classifier Models for Iris and Penguins (use A1 and early tutorials):
  * Logistic Regression (solver='lbfgs')
  * Decision Tree (max_depth=3)
  * KNN (n_neighbors=5)

Any 2 CNN models of your choice for CIFAR-10 dataset (do some research, don't create anything from scratch unless you want to, use the ones provided by libraries and frameworks)

HINT: It is highly recommended that any code from previous assignments and tutorials be reused for tool implementation.

#### IMPORTANT: DON'T FORGET TO IMPORT LIBRARIES REQUIRED FOR ML TASKS!

In [47]:
# Q3a (4 marks): Implement model_memory tool.
# This tool should provide the agent with details about models or datasets
# Example: when asked about Penguin dataset, the agent can use memory to look up
# the source to obtain the dataset.


# YOUR ANSWER GOES HERE

from sklearn.datasets import load_iris
import torchvision.datasets as datasets
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles, make_classification, make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import seaborn as sb

def model_memory(query: str) -> dict:
    memory_db = {
        # Datasets
        "penguins dataset": {
            "source": sb.load_dataset,
            "params": {"name": "penguins"},
            "description": "Tabular dataset containing 344 samples of four physical measurements (bill length/depth, flipper length, and body mass) across three penguin species.",
            "type": "Dataset"
        },

        "iris dataset": {
            "source": load_iris,
            "params": {},
            "description": "Tabular dataset with 150 samples of four numeric measurements (sepal/petal length and width) across three iris species.",
            "type": "Dataset"
        },

        "cifar10 dataset": {
            "source": datasets.CIFAR10,
            "params": {"root": "./data", "train": True, "download": True},
            "description": "Image dataset with 60,000 32x32 color images across 10 distinct classes, split into 50,000 training and 10,000 testing examples.",
            "type": "Dataset"
        },

        # Models
        "logistic regression": {
            "source": LogisticRegression,
            "params": {"solver": "lbfgs"},
            "description": "A linear classifier that predicts probabilities using a logistic function for the Iris dataset.",
            "type": "Model"
        },

        "decision tree": {
            "source": DecisionTreeClassifier,
            "params": {"max_depth": 3},
            "description": "A tree-based model that splits data into branches for the Penguins dataset.",
            "type": "Model"
        },

        "knn": {
            "source": KNeighborsClassifier,
            "params": {"n_neighbors": 5},
            "description": "A nearest-neighbor model that classifies samples based on closest points for the Penguins dataset.",
            "type": "Model"
        },

        "mini-resnet": {
            "source": models.resnet18,
            "params": {"num_classes": 10, "pretrained": False},
            "description": "A residual CNN model used for image classification on CIFAR-10.",
            "type": "Model"
        },

        "simple-cnn": {
            "source": models.mobilenet_v2,
            "params": {"layers": ["Conv2D", "MaxPooling", "Flatten", "Dense"], "pretrained": False},
            "description": "A basic convolutional neural network for image classification on CIFAR-10.",
            "type": "Model"
        }
    }

    query = query.lower().strip()

    for key in memory_db:
        if key in query:
            return memory_db[key]

    return {"error": f"No information found for '{query}'."}

In [48]:
# Q3b (4 marks): Implement dataset_loader tool.
# loads dataset after obtaining info from memory


# YOUR ANSWER GOES HERE

def dataset_loader(dataset_name: str):
    info = model_memory(dataset_name)

    if "error" in info:
        return info["error"]

    load_func = info["source"]
    params = info["params"]

    try:
        dataset = load_func(**params)
        return dataset
    except Exception as e:
        return f"Error loading dataset: {e}"

iris_data = dataset_loader("iris dataset")
print(f"Iris dataset loaded: {iris_data}")

penguins_data = dataset_loader("penguins dataset")
print(f"Penguins dataset loaded, head:\n{penguins_data.head() if hasattr(penguins_data, 'head') else penguins_data}")

cifar10_data = dataset_loader("cifar10 dataset")
print(f"CIFAR-10 dataset loaded: {cifar10_data}")

Iris dataset loaded: {'data': array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 

100%|██████████| 170M/170M [00:02<00:00, 78.0MB/s]


CIFAR-10 dataset loaded: Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train


In [49]:
# Q3c (4 marks): Implement dataset_preprocessing tool.
# preprocesses the dataset to work with the chosen model, and does the splits


# YOUR ANSWER GOES HERE

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

def dataset_preprocessing(data, dataset_name: str):
    dataset_name = dataset_name.lower()

    # Penguins dataset
    if "penguins" in dataset_name:
        # Drop missing values
        data = data.dropna()

        # Select features and target
        X = data.drop("species", axis=1)
        y = data["species"]

        # Convert categorical variables to numeric
        X = pd.get_dummies(X)

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Scale features
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        return X_train, X_test, y_train, y_test

    # Iris dataset
    elif "iris" in dataset_name:
        # Select features and target
        X = data.data
        y = data.target

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Scale features
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        return X_train, X_test, y_train, y_test

    # CIFAR-10 dataset
    elif "cifar" in dataset_name:
        # Already preprocessed (images + labels)
        return data

    return "Unknown dataset"

In [50]:
# Q3d (4 marks): Implement train_model tool.
# trains selected model on selected dataset, the agent should not use this tool
# on datasets and models that cannot work together.


# YOUR ANSWER GOES HERE

In [ ]:
# Q3e (3 marks): Implement evaluate_model tool
# evaluates the models and shows the quality metrics (accuracy, precision, and anything else of your choice)


# YOUR ANSWER GOES HERE

In [ ]:
# Q3f (5 marks): Implement visualize_results tool
# provides results of the training/evaluation, open-ended task (2 plots minimum)


# YOUR ANSWER GOES HERE

In [ ]:
# Q3g (5 marks) Now Provide JSON Schema That Helps LLM Understand your tools, make sure it is valid JSON:

TOOL_SCHEMAS = [
    {
        "name":"",
        "description": "",

        "parameters": {
        # See Tutorial 10
        }
    },
    {
        "name": "",
        "description": "",

        "parameters": {
        # See Tutorial 10
        }
    }
]

# Task 4: System Prompt (10 marks)
A system prompt is essential for guiding an agent's behavior by establishing its purpose, capabilities, tone, and workflow patterns. It acts as the "personality and instruction manual" for the agent, defining the format of interactions (like using Thought/Action/Observation steps in our ML agent), available tools, response styles, and domain-specific knowledge—all while remaining invisible to the end user. This hidden layer of instruction ensures the agent consistently follows the intended reasoning process and operational constraints while providing appropriate and helpful responses, effectively serving as the blueprint for the agent's behavior across all interactions.


In [ ]:
# Q4a (10 marks) Build a system prompt to guide the agent.
# Study Tutorial 10.
# Try to find alternative wording to keep the agent in the desired loop,
# don't just copy the prompt from the tutorial.

# Make sure to reflect proper examples of ReAct loop and Tool use.

# Use the following function:

def create_agent():
    # your system prompt goes inside the multiline string

    tools_json = json.dumps(TOOL_SCHEMAS, indent=2)

    system_prompt = """

    """.strip()

    return ML_Agent(system_prompt)


# Task 5: Set the Agent Loop (16 marks)

Now we are building automation of our Thought/Action/Observation sequence in a single ReAct Loop.


In [ ]:
# Q5a1: (4 marks) Explain why we need the following data structure and function and answer the questions.
# Do not copy-paste tutorial comments
# Fill it in with appropriate values:
KNOWN_ACTIONS = {
   # HINT See Tutorial 10.
}
# Question 1: What happens if no values are provided?


# Question 2: Explain why we need this function and why are strategies suggested in this order?

def extract_json_block(text: str) -> dict | None:
    fence_match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence_match:
        try:
            return json.loads(fence_match.group(1))
        except json.JSONDecodeError:
            pass
    brace_match = re.search(r"(\{.*\})", text, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(1))
        except json.JSONDecodeError:
            pass
    return None


In [ ]:
# Q5a2: (4 marks)

# Question 1: Explain why we need the following data structure and function.
# Question 2: Are these strategies useful in our case?
# Question 3: Does it interfere with actual tool calls? How or why not?

def sanitize_observation(text: str) -> str:
    text = re.sub(r'```json\s*\{.*?\}\s*```', '[REDACTED]', text, flags=re.DOTALL)
    text = re.sub(r'\{"type"\s*:', '[REDACTED: {"type":', text)

    return text


In [ ]:
# Q5b: (6 marks) Explain how the ReAct loop works line by line.
# You are not allowed to copy/paste comments from tutorial 10
# This paper might be helpful: https://react-lm.github.io/

number_of_steps = 5 # adjust this number for your implementation, to avoid an infinite loop

def react_loop(question: str, max_turns: int = number_of_steps) -> List[Dict[str, str]]:
    agent = create_agent()

    next_prompt = question
    for turn in range(max_turns):
        result = agent(next_prompt)
        print(result)

        thought_match = re.search(r"Thought:\s*(.+)", result)
        thought_text = thought_match.group(1).strip() if thought_match else ""

        parsed = extract_json_block(result)

        if parsed is None:
            print(f"\n>>> Final Answer (no JSON detected): {result}")
            break

        msg_type = parsed.get("type")

        if msg_type == "answer":
            answer = parsed.get("content", result)
            print(f"\n>>> Final Answer: {answer}")
            break

        elif msg_type == "action":
            tool_name  = parsed.get("tool")
            tool_input = parsed.get("input", {})

            print(f"INFO: Detected action: {tool_name} with input: {tool_input}")

            if tool_name not in KNOWN_ACTIONS:
                print(f"WARNING: Unknown tool '{tool_name}' — feeding error back.")
                next_prompt = (f"Observation: Error — unknown tool '{tool_name}'. "
                               f"Available tools are: {', '.join(KNOWN_ACTIONS.keys())}")
                continue

            arg_value = list(tool_input.values())[0] if tool_input else ""

            print(f"\n ---> Executing {tool_name}({arg_value})")
            try:
                observation = KNOWN_ACTIONS[tool_name](arg_value)

            except Exception as e:
                tool_schema = next(
                    (t for t in TOOL_SCHEMAS if t["name"] == tool_name), None
                )
                observation = (
                    f"Error executing {tool_name}: {e}\n"
                    f"Hint — expected schema: {json.dumps(tool_schema, indent=2)}"
                )
                print(f"SELF-HEAL: Feeding error + schema back to agent")

            observation = sanitize_observation(str(observation))

            print(f"Observation: {observation}")

            next_prompt = f"Observation: {observation}"

        else:
            print(f"WARNING: Unrecognized JSON type '{msg_type}'")
            next_prompt = (f"Observation: Error — unrecognized response type '{msg_type}'. "
                           f"Use 'action' or 'answer'.")

    else:
        print(f"\nWARNING: Reached maximum turns ({max_turns}) without a final answer.")

    return agent.state.messages

# This is just for semantic clarity
def query(initial_prompt: str, max_turns: int = 5) -> List[Dict[str, str]]:
    return react_loop(initial_prompt, max_turns=max_turns)

In [ ]:
# Q5c: (2 marks)
# QUESTION 1: How can we check the whole raw history of the agent's interaction with LLM?


# Task 6: Run your agent (15 marks)

Let's see if your agent works

In [ ]:
# Execute any THREE example prompts using your agent. (Each working prompt example will give you 5 marks, 5x3=15)
# DONT FORGET TO SAVE THE OUTPUT

# User Prompt examples you should be able to give to your agent:
# **Evaluate Linear Regression Model on Iris Dataset**
# **Train a logistic regression model on the Iris dataset**
# **Load the Penguins dataset and preprocess it.**
# **Train a decision tree model on the Penguins dataset and evaluate it.**
# **Load the CIFAR-10 dataset and train Mini-ResNet CNN, visualize results**

# Use this template (You can have something similar, it is just an example):

# Example 1: Prompt
print("\nExample 1: Evaluate Linear Regression Model on Iris Dataset")
print("=" * 50)
task = "Evaluate Linear Regression Model on Iris Dataset"
result = query(task)
print("\n" + "=" * 50 + "\n")

# Example 2

# Example 3


# Task 7: Own tool (Bonus 10 marks)
Not valid without completion of all the previous tasks and tool implementations.

In [ ]:
# Build your own additional ML-related tool and provide an example of interaction with your reasoning agent
# using a prompt of your choice that makes the agent use your tool at one of the reasoning steps.


# Task 8: Finalizing (Bonus 10 marks)

In [ ]:
# Q8a: Why do you need to call this function? (5 marks)
cleanup_resources()

# Q8b: (5 marks)
# How to check how many tokens were spent?
# Print the cost table.

# Hint: Use llm-connector documentation (https://github.com/isbogdanov/llm_connector)

Good luck!

## Signature:
Don't forget to insert your name and student number and execute the snippet below.



In [ ]:
!pip install watermark
# Provide your Signature:
%load_ext watermark
%watermark -a 'Your Name, #Student_Number' -nmv --packages numpy,pandas,sklearn,matplotlib,seaborn,graphviz,groq,torch